# Keypoint MoSeq for Home Cage SLEAP Data

This notebook uses the home-cage SLEAP organization already used in `Pose_Tracking`.

- TDT trial folders define `subject_name`
- SLEAP `.analysis.h5` files are matched by `subject_name`
- each SLEAP track (`subject`, `agent`) is exported as its own Keypoint MoSeq recording

The helper functions live in `keypoint_moseq_utils.py`.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Pose_Tracking":
    POSE_TRACKING_DIR = PROJECT_ROOT
else:
    POSE_TRACKING_DIR = PROJECT_ROOT / "Pose_Tracking"

if str(POSE_TRACKING_DIR) not in sys.path:
    sys.path.insert(0, str(POSE_TRACKING_DIR))

from keypoint_moseq_utils import (
    build_home_cage_dataset,
    load_kpms_inputs,
    results_to_frame_table,
    save_kpms_inputs,
)


## Paths and Settings

Fill in the directories for your home-cage experiment.

In [ ]:
experiment_dir = Path(r"C:\Users\alber\OneDrive\Desktop\PC_Lab\Photometry\Pilot_2\Combined_Cohorts\Home_Cage\SLEAP\all_tdt")
behavior_dir = Path(r"C:\Users\alber\OneDrive\Desktop\PC_Lab\Photometry\Pilot_2\Combined_Cohorts\Home_Cage\SLEAP\all_csvs")
sleap_dir = Path(r"C:\Users\alber\OneDrive\Desktop\PC_Lab\Photometry\Pilot_2\Combined_Cohorts\Home_Cage\SLEAP\sleap_id_corrected")
video_dir = Path(r"C:\Users\alber\OneDrive\Desktop\PC_Lab\Photometry\Pilot_2\Combined_Cohorts\Home_Cage\SLEAP\HC_Videos_Only")
project_dir = POSE_TRACKING_DIR / "keypoint_moseq_project_home_cage"

fps = 10.0
include_tracks = ("subject", "agent")

# Set True if you want each manual social bout exported as its own recording.
use_bouts = False

# Default bodypart handling for fitting.
exclude_bodyparts = ["Tail_Base"]
anterior_bodyparts = ["Nose"]
posterior_bodyparts = ["Head"]

project_dir.mkdir(parents=True, exist_ok=True)
project_dir


## Export Keypoint MoSeq Inputs

This reads the SLEAP files directly and packages them for Keypoint MoSeq.

In [ ]:
coordinates, confidences, bodyparts, skeleton, metadata_df = build_home_cage_dataset(
    experiment_dir=str(experiment_dir),
    sleap_dir=str(sleap_dir),
    fps=fps,
    include_tracks=include_tracks,
    use_bouts=use_bouts,
    behavior_dir=str(behavior_dir) if use_bouts else None,
)

inputs_path = project_dir / "home_cage_kpms_inputs.pkl"
metadata_path = project_dir / "recording_metadata.csv"

save_kpms_inputs(
    output_path=str(inputs_path),
    coordinates=coordinates,
    confidences=confidences,
    bodyparts=bodyparts,
    skeleton=skeleton,
    metadata_df=metadata_df,
)
metadata_df.to_csv(metadata_path, index=False)

print(f"Saved inputs to: {inputs_path}")
print(f"Saved metadata to: {metadata_path}")
print(f"Recordings exported: {len(coordinates)}")
metadata_df.head()


In [ ]:
sorted(bodyparts), skeleton[:10]


## Review the Exported Recordings

Each recording is a single animal track, optionally split by bout.

In [ ]:
metadata_df.groupby(["track_name", "subject_name"]).agg(
    n_recordings=("recording_name", "count"),
    total_frames=("n_frames", "sum"),
).reset_index().head(20)


## Load Keypoint MoSeq

Run this only after installing `keypoint_moseq` in the Python environment backing this notebook.

In [ ]:
import keypoint_moseq as kpms

payload = load_kpms_inputs(str(inputs_path))
coordinates = payload["coordinates"]
confidences = payload["confidences"]
bodyparts = payload["bodyparts"]
skeleton = payload["skeleton"]
metadata_df = payload["metadata_df"]

print(f"Loaded {len(coordinates)} recordings for Keypoint MoSeq")


## Set Up the Keypoint MoSeq Project

In [ ]:
config_path = project_dir / "config.yml"

if not config_path.exists():
    print(f"No config found at {config_path}. Creating a default config first.")
    kpms.setup_project(
        str(project_dir),
        bodyparts=bodyparts,
        skeleton=skeleton,
        use_bodyparts=bodyparts,
        anterior_bodyparts=anterior_bodyparts,
        posterior_bodyparts=posterior_bodyparts,
        video_dir=str(video_dir),
        fps=fps,
        overwrite=True,
    )
else:
    print(f"Using existing config: {config_path}")

use_bodyparts = [bp for bp in bodyparts if bp not in exclude_bodyparts]
missing_anterior = [bp for bp in anterior_bodyparts if bp not in use_bodyparts]
missing_posterior = [bp for bp in posterior_bodyparts if bp not in use_bodyparts]

if missing_anterior or missing_posterior:
    raise ValueError(
        "Chosen anterior/posterior bodyparts must also be present in use_bodyparts. "
        f"Missing anterior: {missing_anterior}; missing posterior: {missing_posterior}"
    )

kpms.update_config(
    str(project_dir),
    bodyparts=bodyparts,
    skeleton=skeleton,
    video_dir=str(video_dir),
    anterior_bodyparts=anterior_bodyparts,
    posterior_bodyparts=posterior_bodyparts,
    use_bodyparts=use_bodyparts,
    fps=fps,
    latent_dim=10,
)

config = kpms.load_config(str(project_dir))
config


## Optional Outlier Removal

In [ ]:
run_outlier_removal = True

if run_outlier_removal:
    coordinates, confidences = kpms.outlier_removal(
        coordinates,
        confidences,
        str(project_dir),
        overwrite=False,
        **config,
    )


## Format Data and Fit PCA

In [ ]:
data, metadata = kpms.format_data(coordinates, confidences, **config)
pca = kpms.fit_pca(**data, **config)
kpms.save_pca(pca, str(project_dir))
pca


## Fit the Model

The two-stage fit below mirrors the command-line script setup.

In [ ]:
model_name = None
num_ar_iters = 50
num_full_iters = 500

model = kpms.init_model(data, pca=pca, **config)
model = kpms.update_hypparams(model, kappa=1e6)
model, model_name = kpms.fit_model(
    model,
    data,
    metadata,
    str(project_dir),
    model_name=model_name,
    ar_only=True,
    num_iters=num_ar_iters,
)

model, data, metadata, current_iter = kpms.load_checkpoint(
    str(project_dir),
    model_name,
    iteration=num_ar_iters,
)
model = kpms.update_hypparams(model, kappa=1e4)
model = kpms.fit_model(
    model,
    data,
    metadata,
    str(project_dir),
    model_name,
    ar_only=False,
    start_iter=current_iter,
    num_iters=current_iter + num_full_iters,
)[0]

model_name


## Extract Syllables and Save a Frame Table

In [ ]:
kpms.reindex_syllables_in_checkpoint(str(project_dir), model_name)
model, data, metadata, _ = kpms.load_checkpoint(str(project_dir), model_name)
results = kpms.extract_results(model, metadata, str(project_dir), model_name)
kpms.save_results_as_csv(results, str(project_dir), model_name)

syllable_frames = results_to_frame_table(results, metadata_df)
syllable_frames_path = project_dir / model_name / "syllable_frames.csv"
syllable_frames.to_csv(syllable_frames_path, index=False)

print(f"Saved frame table to: {syllable_frames_path}")
syllable_frames.head()


## Quick Summaries

In [ ]:
syllable_frames.groupby(["track_name", "syllable"]).size().reset_index(name="n_frames").sort_values(
    ["track_name", "n_frames"], ascending=[True, False]
).head(20)
